# Getting Started with TeamUP Database Testing

This notebook introduces the TeamUP Python testing framework and demonstrates basic database connectivity.

## Prerequisites

- Python 3.12 installed
- `uv` package manager configured
- PostgreSQL database running on localhost:5432
- `.env` file configured with database credentials

## Import Required Modules

First, let's import all the necessary modules from our teamup package.

In [ ]:
# Add parent directory to Python path to import teamup package
import sys
sys.path.insert(0, '..')

# Import TeamUP modules
from teamup.database import get_session, test_connection
from teamup.models import Employee, LearningMaterialType, LearningMaterial
from teamup.repository import EmployeeRepository, LearningMaterialTypeRepository, LearningMaterialRepository
from teamup.config import DATABASE_URL, DB_NAME

print("✓ All modules imported successfully")

## Test Database Connection

Let's verify that we can connect to the PostgreSQL database.

In [ ]:
# Test connection
if test_connection():
    print(f"✓ Successfully connected to database: {DB_NAME}")
else:
    print("✗ Failed to connect to database")

## Execute a Simple Query

Let's run a simple query to verify everything works.

In [ ]:
from sqlalchemy import text

with get_session() as session:
    result = session.execute(text("SELECT version()"))
    version = result.scalar()
    print(f"PostgreSQL version: {version}")

## Check Current Data Counts

Let's see how many records we have in each table.

In [ ]:
with get_session() as session:
    emp_repo = EmployeeRepository(session)
    type_repo = LearningMaterialTypeRepository(session)
    material_repo = LearningMaterialRepository(session)
    
    print(f"Employees: {emp_repo.count()}")
    print(f"Learning Material Types: {type_repo.count()}")
    print(f"Learning Materials: {material_repo.count()}")

## Understanding the Context Manager Pattern

The `get_session()` function is a context manager that:

1. **Automatically opens** a database connection
2. **Sets the correct schema** (teamup)
3. **Commits on success** - all changes are saved
4. **Rolls back on error** - nothing is saved if an exception occurs
5. **Closes the connection** - resources are cleaned up

Example:

In [ ]:
# Example: This will be rolled back due to the exception
try:
    with get_session() as session:
        repo = EmployeeRepository(session)
        # This would create an employee
        emp = repo.create(fullname="Temporary Employee", age=25)
        print(f"Created employee with ID: {emp.id}")
        
        # But this exception causes a rollback
        raise ValueError("Oops! This causes a rollback")
except ValueError as e:
    print(f"Exception caught: {e}")

# Verify the employee was NOT created (rolled back)
with get_session() as session:
    repo = EmployeeRepository(session)
    found = repo.find_by_name("Temporary Employee")
    print(f"Employee found after rollback: {len(found)}")

## Next Steps

Now that you've confirmed the connection works, check out:

- **02_employee_crud.ipynb** - Learn how to create, read, update, and delete employees
- **03_learning_material_crud.ipynb** - Work with learning materials and types
- **04_advanced_examples.ipynb** - Complex queries, JSON handling, and workflows